# EY AI & Data Challenge — Ensemble Method 


## 1. Imports

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

import xgboost as xgb 
from xgboost.sklearn import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

from datetime import date
import os

import pandas as pd
import numpy as np


print('All imports successful')

## 2. Load Data

In [ ]:
wq_data = pd.read_csv("complete_data.csv")

## 3. Define Feature Sets

**RULE**: No water quality parameters as inputs. Must be computable for any unseen location.

In [ ]:
wq_data.columns

In [ ]:
# Total Alkalinity - driven by geology, temperature, evaporation
FEATURES_TA = [
    
      'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'
        
]

# Electrical Conductance - driven by salinity, evaporation, climate
FEATURES_EC = [
    
       'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'
      
]

# Dissolved Reactive Phosphorus - driven by runoff, agriculture, season
FEATURES_DRP = [
   
       'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'

               
]

# Verify no target leakage
target = ['Total Alkalinity','Electrical Conductance',
             'Dissolved Reactive Phosphorus']
for name, feats in [('TA',FEATURES_TA),('EC',FEATURES_EC),('DRP',FEATURES_DRP)]:
    leaks = [f for f in feats if f in target]
    print(f'{name}: {"CLEAN" if not leaks else "LEAKING: "+str(leaks)}')
    print(f'  {len(feats)} features')

In [ ]:
wq_data.columns

## 4. Base Models Definition

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

def build_base_models():
    """
    Three diverse models that make different kinds of errors.
    Diversity is what makes ensembling work.
    
    """
    return {
        'xgboost': XGBRegressor(
            max_depth=6,
            n_estimators=200,
            learning_rate=0.01,
        ),
        'etr':  ExtraTreesRegressor(
            n_estimators=200,
            
        ),
        'random_forest': RandomForestRegressor(
            n_estimators=200,
            max_depth=47,
            min_samples_leaf=10,
            max_features=0.6,
        ),
    }

print('Base models defined: XGBoost, ExtraTreeRegressor, RandomForest')

## 5. Level 1 — Simple Average Ensemble 

In [ ]:
def simple_average_ensemble(X, y, features, param_name,
                              test_size=0.3):

    from sklearn.model_selection import train_test_split

    print(f"\n{'='*60}")
    print(f'Level 1: Simple Average Ensemble — {param_name}')
    print(f"{'='*60}")

    X_feats = X[features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_feats, y, test_size=test_size, random_state=42
    )

    models     = build_base_models()
    fold_preds = {}
    model_r2   = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        fold_preds[name] = pred
        r2 = r2_score(y_test, pred)
        model_r2[name] = r2
        print(f'  {name:15s} R²: {r2:.3f}')

    ensemble_pred = np.mean(list(fold_preds.values()), axis=0)
    ensemble_r2   = r2_score(y_test, ensemble_pred)
    print(f'  {"ensemble":15s} R²: {ensemble_r2:.3f}  <-- target')
    # calculate average r2 score of 3 models

    return models, model_r2, ensemble_r2


In [ ]:

X_all = wq_data.copy()
y_TA  = wq_data['Total Alkalinity']
y_EC  = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

_, _, r2_TA_l1  = simple_average_ensemble(X_all, y_TA,  FEATURES_TA,  'Total Alkalinity')
_, _, r2_EC_l1  = simple_average_ensemble(X_all, y_EC,  FEATURES_EC,  'Electrical Conductance')
_, _, r2_DRP_l1 = simple_average_ensemble(X_all, y_DRP, FEATURES_DRP, 'Dissolved Reactive Phosphorus')

print(f"\n{'='*60}")
print('LEVEL 1 SUMMARY')
print(f"{'='*60}")
print(f'TA  ensemble R²: {r2_TA_l1:.3f}')
print(f'EC  ensemble R²: {r2_EC_l1:.3f}')
print(f'DRP ensemble R²: {r2_DRP_l1:.3f}')
print(f'Mean R²: {np.mean([r2_TA_l1, r2_EC_l1, r2_DRP_l1]):.3f}')


## 6. Level 2 — Stacking Ensemble

A meta-model learns the optimal combination of base model predictions.

In [ ]:
def stacking_ensemble(X, y, features, param_name,
                        test_size=0.3, n_cv_folds=5):
    """
    Level 2: Stacking with k-fold CV to generate out-of-fold predictions.
    Meta-model learns optimal combination of base model predictions.
    """
    from sklearn.model_selection import KFold

    print(f"\n{'='*60}")
    print(f'Level 2: Stacking Ensemble — {param_name}')
    print(f"{'='*60}")

    X_feats   = X[features].reset_index(drop=True)
    y_reset   = y.reset_index(drop=True)
    
    models    = build_base_models()
    n_models  = len(models)
    n_samples = len(X_feats)

    # Out-of-fold predictions matrix: (n_samples, n_models)
    oof_preds = np.zeros((n_samples, n_models))

    # Generate OOF predictions using KFold CV
    print(f'Generating out-of-fold predictions ({n_cv_folds} folds)...')
    kf = KFold(n_splits=n_cv_folds, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_feats)):
        X_tr = X_feats.iloc[train_idx]
        X_te = X_feats.iloc[test_idx]
        y_tr = y_reset.iloc[train_idx]

        for i, (name, model) in enumerate(models.items()):
            model.fit(X_tr, y_tr)
            oof_preds[test_idx, i] = model.predict(X_te)

        print(f'  Fold {fold+1} complete')

    # Train meta-model on OOF predictions
    print('\nTraining meta-model (Ridge)...')
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(oof_preds, y_reset)

    oof_r2 = r2_score(y_reset, meta_model.predict(oof_preds))
    print(f'Meta-model OOF R²: {oof_r2:.3f}')

    model_names = list(models.keys())
    print('\nMeta-model coefficients:')
    for name, coef in zip(model_names, meta_model.coef_):
        print(f'  {name:15s}: {coef:.4f}')

    # Retrain ALL base models on full data for submission
    print('\nRetraining on full training data...')
    final_models = {}
    for name, model in build_base_models().items():
        model.fit(X_feats, y_reset)
        final_models[name] = model
        print(f'  {name} done')

    return final_models, meta_model, model_names, oof_r2


In [ ]:
final_models_TA,  meta_TA,  names_TA,  oof_r2_TA  = stacking_ensemble(X_all, y_TA,  FEATURES_TA,  'Total Alkalinity')
final_models_EC,  meta_EC,  names_EC,  oof_r2_EC  = stacking_ensemble(X_all, y_EC,  FEATURES_EC,  'Electrical Conductance')
final_models_DRP, meta_DRP, names_DRP, oof_r2_DRP = stacking_ensemble(X_all, y_DRP, FEATURES_DRP, 'Dissolved Reactive Phosphorus')

print(f"\n{'='*60}")
print('STACKING SUMMARY')
print(f"{'='*60}")
print(f'TA  OOF R²: {oof_r2_TA:.3f}')
print(f'EC  OOF R²: {oof_r2_EC:.3f}')
print(f'DRP OOF R²: {oof_r2_DRP:.3f}')
print(f'Mean:       {np.mean([oof_r2_TA, oof_r2_EC, oof_r2_DRP]):.3f}')


## 7. Compare All Three Levels

In [ ]:
print(f"{'='*60}")
print('ENSEMBLE LEVEL COMPARISON')
print(f"{'='*60}")
print(f'{"Parameter":30s} {"L1 Avg":>10} {"L2 Stack":>10}')
print('-'*60)

for param, l1, l2 in [
    ('Total Alkalinity',        r2_TA_l1, oof_r2_TA),
    ('Electrical Conductance',  r2_EC_l1,  oof_r2_EC),
    ('Dissolved React. Phos.',  r2_DRP_l1, oof_r2_DRP),
]:
    best = max(l1, l2)
    def mark(v): return f'{v:.3f} *' if v == best else f'{v:.3f}  '
    print(f'{param:30s} {mark(l1):>12} {mark(l2):>12}')

print(f"{'='*60}")
print('* = best level for that parameter')
print('Use the starred level for each parameter in submission')


## 8. Load & Prepare Validation Data

In [ ]:
val_data = pd.read_csv("complete_val_data.csv")
print(val_data.shape)

In [ ]:
# Total Alkalinity - driven by geology, temperature, evaporation
FEATURES_TA = [
    
     'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'
     
        
      
]

# Electrical Conductance - driven by salinity, evaporation, climate
FEATURES_EC = [
    
       'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'
      
       
]

# Dissolved Reactive Phosphorus - driven by runoff, agriculture, season
FEATURES_DRP = [
   
       'green', 'swir22', 'NDMI', 'MNDWI', 'NDVI',
       'NDBI', 'NDTI',  'pet', 'ppt', 'tmax',  'def', 'q',
       'vpd', 'ws', 'soil', 
       'season_autumn', 'season_spring', 'season_summer', 'season_winter',
       'Month_sin', 'Month_cos'
      
        
       
]

# Verify no target leakage
FORBIDDEN = ['Total Alkalinity','Electrical Conductance',
             'Dissolved Reactive Phosphorus']
for name, feats in [('TA',FEATURES_TA),('EC',FEATURES_EC),('DRP',FEATURES_DRP)]:
    leaks = [f for f in feats if f in FORBIDDEN]
    print(f'{name}: {"CLEAN" if not leaks else "LEAKING: "+str(leaks)}')
    print(f'  {len(feats)} features')

## 9. Generate Submission Predictions

Using Level 3 Stacking (change to Level 1 or 2 if stacking scores lower)

In [ ]:
def predict_stacking(final_models, meta_model, 
                     model_names, X_val, features):
    """
    Generate stacking predictions for validation data.
    """
    # Get base model predictions
    base_preds = np.column_stack([
        model.predict(X_val[features])
        for model in final_models.values()
    ])

    print('Base model prediction ranges:')
    for i, name in enumerate(model_names):
        print(f'  {name:15s}: '
              f'min={base_preds[:,i].min():.1f}  '
              f'max={base_preds[:,i].max():.1f}  '
              f'mean={base_preds[:,i].mean():.1f}')

    # Meta-model combines base predictions
    final_pred = meta_model.predict(base_preds)
    final_pred = np.clip(final_pred, 0, None)

    return final_pred

In [ ]:
# Generate predictions using stacking ensemble
print('Generating predictions...')

pred_TA = predict_stacking(
    final_models_TA, meta_TA, names_TA, val_data, FEATURES_TA
)
pred_EC = predict_stacking(
    final_models_EC, meta_EC, names_EC, val_data, FEATURES_EC
)
pred_DRP = predict_stacking(
    final_models_DRP, meta_DRP, names_DRP, val_data, FEATURES_DRP
)

# Clip negatives - water quality cannot be negative
pred_TA  = np.clip(pred_TA,  0, None)
pred_EC  = np.clip(pred_EC,  0, None)
pred_DRP = np.clip(pred_DRP, 0, None)

## 11. Sanity Check — Prediction Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, train_vals, val_preds) in zip(axes, [
    ('Total Alkalinity',              y_TA,  pred_TA),
    ('Electrical Conductance',        y_EC,  pred_EC),
    ('Dissolved Reactive Phosphorus', y_DRP, pred_DRP),
]):
    ax.hist(train_vals, bins=40, alpha=0.6, label='Training actual',       color='steelblue')
    ax.hist(val_preds,  bins=40, alpha=0.6, label='Validation predicted',  color='orange')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Training Distribution vs Validation Predictions', fontsize=12)
plt.tight_layout()
plt.show()
print('If orange bars look wildly different from blue, investigate your pipeline.')

## 10. Save & Upload Submission

In [ ]:
submission_df = pd.DataFrame({
    'Longitude':                       val_data['Longitude'].values,
    'Latitude':                        val_data['Latitude'].values,
    'Sample Date':                     val_data['Sample Date'].values,
    'Total Alkalinity':                pred_TA,
    'Electrical Conductance':          pred_EC,
    'Dissolved Reactive Phosphorus':   pred_DRP
})

print(f'Submission shape: {submission_df.shape}')
display(submission_df.head())

submission_df.to_csv('/tmp/submission_ensemble_modified.csv', index=False)

session.sql("""
    PUT file:///tmp/submission_ensemble_modified.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print('Submission saved and uploaded.')
print('Refresh browser to see file in sidebar.')

In [ ]:
# ── Load files ──────────────────────────────────────────────────────────────
actual = pd.read_csv('submission_extracted.csv')   # ground truth
pred   = pd.read_csv('submission_ensemble_modified.csv')     # model predictions
 
# ── Standardise coordinates & dates for merging ─────────────────────────────
for df in [actual, pred]:
    df['lat_r']      = df['Latitude'].round(3)
    df['lon_r']      = df['Longitude'].round(3)
    df['Sample Date'] = pd.to_datetime(df['Sample Date'])
 
# ── Merge on location + date ─────────────────────────────────────────────────
merged = actual.merge(pred, on=['lat_r', 'lon_r', 'Sample Date'],
                      suffixes=('_actual', '_pred'))
print(f"Rows matched: {len(merged)} / {len(actual)}\n")
 


In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
targets = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus',
]
 
results = []
for t in targets:
    y_true = merged[f'{t}_actual']
    y_pred = merged[f'{t}_pred']
 
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
 
    results.append({'Target': t, 'R²': round(r2, 4),
                     'RMSE': round(rmse, 4)})
 
    print(f"{t}")
    print(f"  R²   = {r2:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")
 
# ── Overall mean R² (competition metric) ────────────────────────────────────
mean_r2 = np.mean([r['R²'] for r in results])
print(f"Mean R² (all targets): {mean_r2:.4f}")